# ترجمة الكلمات الجديدة

دفترٌ صغير، غرضه واحد: يأخذ `pending.json` ويعيد `translated.json`.

**لا تُرفع القاعدة ولا تُنزَّل** — بضعة كيلوبايتات في الاتجاهين بدل ١٢٣ ميغابايت.

| الخطوة | أين |
|---|---|
| `python sync_cards.py --pending` | جهازك |
| ارفع `pending.json` وشغّل هنا | Colab |
| `python sync_cards.py --apply translated.json --push` | جهازك |

In [ ]:
!pip -q install transformers sentencepiece sacremoses
print('تمّ')

### ارفع `pending.json` من أيقونة المجلد على اليسار، ثم شغّل

In [ ]:
import json, os, re, time, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL   = 'facebook/nllb-200-distilled-600M'
SRC, TGT = 'eng_Latn', 'arb_Arab'
BATCH   = 24
ARABIC  = re.compile('[\u0600-\u06ff]')

data = json.load(open('pending.json', encoding='utf-8'))
jobs = data['jobs']
print(f'{len(jobs):,} نصاً')

gpu = torch.cuda.is_available()
tok = AutoTokenizer.from_pretrained(MODEL, src_lang=SRC)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
model = model.to('cuda' if gpu else 'cpu').eval()

# تثبيت اللغة الهدف — بدونه يختار النموذج لغةً عشوائية
bos = tok.convert_tokens_to_ids(TGT)
assert bos and bos != tok.unk_token_id, 'رمز اللغة غير معروف'
print(f"{'كرت رسومي' if gpu else 'معالج'} · الهدف {TGT} ({bos})")

t0, bad = time.time(), 0
for i in range(0, len(jobs), BATCH):
    chunk = jobs[i:i + BATCH]
    enc = tok([j['en'] for j in chunk], return_tensors='pt',
              padding=True, truncation=True, max_length=256)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, num_beams=1,
                             forced_bos_token_id=bos)
    for j, ar in zip(chunk, tok.batch_decode(out, skip_special_tokens=True)):
        ar = (ar or '').strip()
        # حارس: ما ليس عربياً لا يُرسَل — أنتج النموذج مرّةً هندية ورومانية
        if ar and ARABIC.search(ar):
            j['ar'] = ar
        else:
            bad += 1
    el = time.time() - t0
    n = i + len(chunk)
    print(f'  {n:,}/{len(jobs):,} · مضى {el/60:.0f} د · '
          f'بقي ~{(len(jobs)-n)/max(n/el,.1)/60:.0f} د', flush=True)

json.dump(data, open('translated.json', 'w', encoding='utf-8'),
          ensure_ascii=False)
ok = sum(1 for j in jobs if j.get('ar'))
print(f"\nتُرجم {ok:,} · رُفض {bad:,} · "
      f"{os.path.getsize('translated.json')/1024:.0f} ك.ب")

### عيّنة — اقرأها قبل التنزيل

In [ ]:
for j in [x for x in jobs if x.get('ar')][:10]:
    print(f"  {j['en'][:56]}\n     {j['ar']}\n")

In [ ]:
from google.colab import files
files.download('translated.json')